In [ ]:
# 1. Проверяем GPU и ставим окружение
!nvidia-smi
!pip install torch torchvision torchaudio onnx onnxruntime numpy --quiet

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"--> Training on device: {device}")

# ==========================================
# 1. MODEL 1: Spatial Transformer (STN) + ResNet for Air-Writing
# ==========================================
class SpatialTransformer1D(nn.Module):
    """Автоматически выравнивает наклон, масштаб и сдвиг буквы в воздухе."""
    def __init__(self):
        super().__init__()
        self.localization = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(True),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(True),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(True),
            nn.Linear(32, 4) # 2x2 affine matrix parameters
        )
        # Инициализация единичной матрицей
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 1], dtype=torch.float))

    def forward(self, x):
        # x shape: (B, 6, 63) -> features 0 and 1 are dx, dy
        batch_size = x.size(0)
        theta = self.localization(x).view(batch_size, 64)
        theta = self.fc_loc(theta).view(batch_size, 2, 2)
        
        coords = x[:, :2, :] # (B, 2, 63)
        transformed_coords = torch.bmm(theta, coords)
        
        # Обновляем признаки траектории
        out = torch.cat([transformed_coords, x[:, 2:, :]], dim=1)
        return out

class STNStrokeClassifier(nn.Module):
    def __init__(self, num_classes=95):
        super().__init__()
        self.stn = SpatialTransformer1D()
        
        self.backbone = nn.Sequential(
            nn.Conv1d(6, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.SiLU(),
        )
        self.gru = nn.GRU(256, 128, num_layers=2, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Input shape: (B, 63, 6)
        x = x.transpose(1, 2) # (B, 6, 63)
        x = self.stn(x)
        x = self.backbone(x)
        x = x.transpose(1, 2) # (B, T, C)
        out_gru, _ = self.gru(x)
        # Global context pooling
        pooled = torch.mean(out_gru, dim=1)
        logits = self.classifier(pooled)
        return logits

# ==========================================
# 2. MODEL 2: Zero-Latency Hand Gesture MLP
# ==========================================
class HandGestureMLP(nn.Module):
    """Классификатор жестов по 63 координатам ладони (Pinch, Fist, Swipe, Peace, Neutral)"""
    def __init__(self, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

print("--> Architectures compiled successfully. Ready for GPU training.")